In [1]:
# ================================================================
#  LCCN-R3 - McNEMAR ON STORED PREDICTIONS  [CPU, minutes]
#  Replaces/supplements the paired t-test with McNemar's test
#  (continuity correction) on per-comment correctness, reporting the
#  discordant counts b (A correct, B wrong) and c (A wrong, B correct)
#  for the three claim-carrying comparisons:
#    raw vs indicsoundex      (NB6 stored predictions, seed-last)
#    raw vs lccn              (NB6 stored predictions, seed-last)
#    raw vs lccn_indicxlit    (NB4 stored predictions, seed 42)
#  Add Input: paper1-nb6-lccn-xlm43-44 (output),
#             paper1-nb4-lccn-xlmroberta (output).
#  Accelerator None. Save & Run All (Commit).
# ================================================================
import glob, os, sys
import pandas as pd
from scipy.stats import chi2
print("python", sys.version.split()[0])
print("prediction files found:")
for f in sorted(glob.glob("/kaggle/input/**/preds/*.csv", recursive=True)):
    print("  ", f)


python 3.12.13
prediction files found:
   /kaggle/input/notebooks/tejaswinina/paper1-nb4-lccn-xlmroberta/preds/XLM-RoBERTa__indicsoundex.csv
   /kaggle/input/notebooks/tejaswinina/paper1-nb4-lccn-xlmroberta/preds/XLM-RoBERTa__indicsoundex_guard.csv
   /kaggle/input/notebooks/tejaswinina/paper1-nb4-lccn-xlmroberta/preds/XLM-RoBERTa__keel.csv
   /kaggle/input/notebooks/tejaswinina/paper1-nb4-lccn-xlmroberta/preds/XLM-RoBERTa__kope.csv
   /kaggle/input/notebooks/tejaswinina/paper1-nb4-lccn-xlmroberta/preds/XLM-RoBERTa__lccn.csv
   /kaggle/input/notebooks/tejaswinina/paper1-nb4-lccn-xlmroberta/preds/XLM-RoBERTa__lccn_base.csv
   /kaggle/input/notebooks/tejaswinina/paper1-nb4-lccn-xlmroberta/preds/XLM-RoBERTa__lccn_indicxlit.csv
   /kaggle/input/notebooks/tejaswinina/paper1-nb4-lccn-xlmroberta/preds/XLM-RoBERTa__lccn_itrans.csv
   /kaggle/input/notebooks/tejaswinina/paper1-nb4-lccn-xlmroberta/preds/XLM-RoBERTa__metaphone.csv
   /kaggle/input/notebooks/tejaswinina/paper1-nb4-lccn-xlmroberta/

In [2]:
# ================================================================
#  CELL 1 - LOAD, ALIGN BY COMMENT ID, McNEMAR
# ================================================================
def load_pred(arm, prefer):
    hits = [f for f in glob.glob(
        f"/kaggle/input/**/preds/XLM-RoBERTa__{arm}.csv", recursive=True)]
    assert hits, f"missing predictions for {arm}"
    pick = next((h for h in hits if prefer in h), hits[0])
    d = pd.read_csv(pick, dtype=str, keep_default_na=False)
    d.columns = [c.strip().lower() for c in d.columns]
    idc  = next(c for c in d.columns if c in ("id", "comment_id"))
    true = next(c for c in d.columns if "true" in c or c == "gold")
    pred = next(c for c in d.columns if "pred" in c)
    d = d[[idc, true, pred]].copy()
    d.columns = ["id", "true", "pred"]
    d["ok"] = (d["true"].str.strip().str.lower()
               == d["pred"].str.strip().str.lower()).astype(int)
    print(f"{arm:<16} {len(d):,} rows  acc {d.ok.mean():.4f}  <- {pick}")
    return d.set_index("id")

def mcnemar(A, B, name_a, name_b):
    j = A.join(B, lsuffix="_a", rsuffix="_b", how="inner")
    assert len(j) == 6000, f"alignment produced {len(j)} rows, expected 6,000"
    assert (j["true_a"] == j["true_b"]).all(), "true labels disagree across files"
    b = int(((j.ok_a == 1) & (j.ok_b == 0)).sum())
    c = int(((j.ok_a == 0) & (j.ok_b == 1)).sum())
    stat = (abs(b - c) - 1) ** 2 / (b + c) if (b + c) else 0.0
    p = float(chi2.sf(stat, 1))
    print(f"\n{name_a} vs {name_b}")
    print(f"  {name_a} correct / {name_b} wrong  (b) = {b:,}")
    print(f"  {name_a} wrong / {name_b} correct  (c) = {c:,}")
    print(f"  McNemar chi2 (continuity corr.) = {stat:.2f}   p = {p:.5f}")

raw6  = load_pred("raw",            "nb6")
inds  = load_pred("indicsoundex",   "nb6")
lccn  = load_pred("lccn",           "nb6")
raw4  = load_pred("raw",            "nb4")
comp  = load_pred("lccn_indicxlit", "nb4")

mcnemar(raw6, inds, "raw", "indicsoundex")
mcnemar(raw6, lccn, "raw", "lccn")
mcnemar(raw4, comp, "raw", "lccn_indicxlit")
print("\nNB6 files are the seed-last stored runs and the NB4 files are the")
print("seed-42 runs, matching the convention already stated in the paper.")
print("DONE - transcribe b, c, chi2, and p from the committed log.")


raw              6,000 rows  acc 0.8023  <- /kaggle/input/notebooks/tejaswinina/paper1-nb6-lccn-xlm43-44/preds/XLM-RoBERTa__raw.csv
indicsoundex     6,000 rows  acc 0.7843  <- /kaggle/input/notebooks/tejaswinina/paper1-nb6-lccn-xlm43-44/preds/XLM-RoBERTa__indicsoundex.csv
lccn             6,000 rows  acc 0.8060  <- /kaggle/input/notebooks/tejaswinina/paper1-nb6-lccn-xlm43-44/preds/XLM-RoBERTa__lccn.csv
raw              6,000 rows  acc 0.8065  <- /kaggle/input/notebooks/tejaswinina/paper1-nb4-lccn-xlmroberta/preds/XLM-RoBERTa__raw.csv
lccn_indicxlit   6,000 rows  acc 0.8145  <- /kaggle/input/notebooks/tejaswinina/paper1-nb4-lccn-xlmroberta/preds/XLM-RoBERTa__lccn_indicxlit.csv

raw vs indicsoundex
  raw correct / indicsoundex wrong  (b) = 478
  raw wrong / indicsoundex correct  (c) = 370
  McNemar chi2 (continuity corr.) = 13.50   p = 0.00024

raw vs lccn
  raw correct / lccn wrong  (b) = 264
  raw wrong / lccn correct  (c) = 286
  McNemar chi2 (continuity corr.) = 0.80   p = 0.37055

r